## This notebook is for cleaning and filtering my streaming data

In [1]:
import pandas as pd
from datetime import time

In [10]:
today = pd.to_datetime("today").date()
streaming = pd.read_csv(f"../data/livesports_schedule_{today}.csv")

In [3]:
streaming.head()

,Date,Time,League,Matchup,Services
0,2026-03-12,7:00 PM,NBA,Detroit Pistons @ Philadelphia 76ers @,['Prime Video']
1,2026-03-12,7:00 PM,NBA,Indiana Pacers @ Phoenix Suns @,"['NBA League Pass', 'FanDuel Sports Network In..."
2,2026-03-12,7:00 PM,NBA,Orlando Magic @ Washington Wizards @,"['NBA League Pass', 'FanDuel Sports Network Fl..."
3,2026-03-12,7:30 PM,NBA,Atlanta Hawks @ Brooklyn Nets @,"['NBA League Pass', 'FanDuel Sports Network So..."
4,2026-03-12,7:30 PM,NBA,Miami Heat @ Milwaukee Bucks @,"['NBA League Pass', 'FanDuel Sports Network Su..."


In [11]:
# convert Time (7:00 PM) to 24-hour format (19:00)
streaming["Time"] = pd.to_datetime(streaming["Time"], format="%I:%M %p").dt.strftime("%H:%M")


In [12]:
streaming["Matchup"] = streaming["Matchup"].str.strip(" @").str.replace(" @ ", ",")

In [13]:
streaming.head()

,Date,Time,League,Matchup,Services
0,2026-03-12,19:00,NBA,"Detroit Pistons,Philadelphia 76ers",['Prime Video']
1,2026-03-12,19:00,NBA,"Indiana Pacers,Phoenix Suns","['NBA League Pass', 'FanDuel Sports Network In..."
2,2026-03-12,19:00,NBA,"Orlando Magic,Washington Wizards","['NBA League Pass', 'FanDuel Sports Network Fl..."
3,2026-03-12,19:30,NBA,"Atlanta Hawks,Brooklyn Nets","['NBA League Pass', 'FanDuel Sports Network So..."
4,2026-03-12,19:30,NBA,"Miami Heat,Milwaukee Bucks","['NBA League Pass', 'FanDuel Sports Network Su..."


In [14]:
streaming_exploded = streaming.explode("Services")
streaming_exploded.head()

,Date,Time,League,Matchup,Services
0,2026-03-12,19:00,NBA,"Detroit Pistons,Philadelphia 76ers",['Prime Video']
1,2026-03-12,19:00,NBA,"Indiana Pacers,Phoenix Suns","['NBA League Pass', 'FanDuel Sports Network In..."
2,2026-03-12,19:00,NBA,"Orlando Magic,Washington Wizards","['NBA League Pass', 'FanDuel Sports Network Fl..."
3,2026-03-12,19:30,NBA,"Atlanta Hawks,Brooklyn Nets","['NBA League Pass', 'FanDuel Sports Network So..."
4,2026-03-12,19:30,NBA,"Miami Heat,Milwaukee Bucks","['NBA League Pass', 'FanDuel Sports Network Su..."


In [85]:
services = streaming_exploded["Services"].unique().tolist()
services_set = set(services)
available_in_detroit = {
        # National Cable & Broadcast Networks
        'ABC', 'CBS', 'NBC', 'Telemundo', 'USA Network',
        'ESPN', 'ESPN2', 'ESPN U', 'ESPNews', 'ESPN App',
        'CBS Sports Network', 'Big Ten Network', 'SEC Network', 'ACC Network',
        'NBA TV', 'NBA League Pass', 'NBC Sports Network', # NBCSN is defunct, but was national
        
        # National Streaming Platforms
        'Peacock', 'Prime Video', 'Paramount+', 'Fubo Sports',
        'ESPN Select', 'ESPN Unlimited', 'NEC Front Row',
        
        # Local to Detroit / Michigan
        'FanDuel Sports Network Detroit Extra',
        'Detroit SportsNet'
    }
print("Available in Detroit:", services_set.intersection(available_in_detroit))

Available in Detroit: {'NBC Sports Network', 'Prime Video', 'Fubo Sports', 'USA Network', 'ESPN Unlimited', 'NBA League Pass', 'Peacock', 'ESPN Select', 'ESPN2', 'Big Ten Network', 'SEC Network'}


In [86]:
streaming_exploded_filterd = streaming_exploded[streaming_exploded["Services"].isin(available_in_detroit)]


In [87]:
# unexplode the Services column to return to a list
streaming_cleaned = streaming_exploded_filterd.groupby(streaming_exploded_filterd.index).agg({
    'Date': 'first',
    'Time': 'first',
    'League': 'first',
    'Matchup': 'first',
    'Services': lambda x: list(x)
}).reset_index(drop=True)
streaming_cleaned.head()

,Date,Time,League,Matchup,Services
0,3/12/2026,19:00,NBA,"Philadelphia 76ers,Detroit Pistons",[Prime Video]
1,3/12/2026,19:00,NBA,"Phoenix Suns,Indiana Pacers",[NBA League Pass]
2,3/12/2026,19:00,NBA,"Washington Wizards,Orlando Magic",[NBA League Pass]
3,3/12/2026,19:30,NBA,"Brooklyn Nets,Atlanta Hawks",[NBA League Pass]
4,3/12/2026,19:30,NBA,"Milwaukee Bucks,Miami Heat",[NBA League Pass]


In [88]:
streaming_cleaned['DayOfWeek'] = pd.to_datetime(streaming_cleaned['Date']).dt.day_name()
streaming_cleaned.head()

,Date,Time,League,Matchup,Services,DayOfWeek
0,3/12/2026,19:00,NBA,"Philadelphia 76ers,Detroit Pistons",[Prime Video],Thursday
1,3/12/2026,19:00,NBA,"Phoenix Suns,Indiana Pacers",[NBA League Pass],Thursday
2,3/12/2026,19:00,NBA,"Washington Wizards,Orlando Magic",[NBA League Pass],Thursday
3,3/12/2026,19:30,NBA,"Brooklyn Nets,Atlanta Hawks",[NBA League Pass],Thursday
4,3/12/2026,19:30,NBA,"Milwaukee Bucks,Miami Heat",[NBA League Pass],Thursday


In [89]:
def filter_prime_time_games(df):
    """
    Filters a DataFrame of games for specific viewing windows:
    - Weekdays (Mon-Fri): 5:00 PM to 11:00 PM
    - Weekends (Sat-Sun): 12:00 PM (Noon) to 11:00 PM
    """
    
    # 1. Convert the 'Time' column to actual datetime time objects
    # Assumes your time is formatted as "HH:MM" (e.g., "19:00")
    game_times = pd.to_datetime(df['Time'], format='%H:%M').dt.time
    
    # 2. Define our time boundaries
    weekday_start = time(17, 0)  # 5:00 PM
    weekend_start = time(12, 0)  # 12:00 PM (Noon)
    end_time = time(23, 0)       # 11:00 PM
    
    # 3. Categorize the days
    weekdays = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']
    weekends = ['Saturday', 'Sunday']
    
    # 4. Create boolean masks for day type
    is_weekday = df['DayOfWeek'].isin(weekdays)
    is_weekend = df['DayOfWeek'].isin(weekends)
    
    # 5. Create boolean masks for valid times
    valid_weekday_time = (game_times >= weekday_start) & (game_times <= end_time)
    valid_weekend_time = (game_times >= weekend_start) & (game_times <= end_time)
    
    # 6. Combine the logic: (Is a Weekday AND Valid Time) OR (Is a Weekend AND Valid Time)
    final_mask = (is_weekday & valid_weekday_time) | (is_weekend & valid_weekend_time)
    
    # 7. Return the filtered dataframe
    return df[final_mask].copy()

# --- Usage Example ---
# Assuming your dataframe is named 'schedule_df'
streaming_cleaned_prime_time = filter_prime_time_games(streaming_cleaned)
streaming_cleaned_prime_time.head()

,Date,Time,League,Matchup,Services,DayOfWeek
0,3/12/2026,19:00,NBA,"Philadelphia 76ers,Detroit Pistons",[Prime Video],Thursday
1,3/12/2026,19:00,NBA,"Phoenix Suns,Indiana Pacers",[NBA League Pass],Thursday
2,3/12/2026,19:00,NBA,"Washington Wizards,Orlando Magic",[NBA League Pass],Thursday
3,3/12/2026,19:30,NBA,"Brooklyn Nets,Atlanta Hawks",[NBA League Pass],Thursday
4,3/12/2026,19:30,NBA,"Milwaukee Bucks,Miami Heat",[NBA League Pass],Thursday


In [90]:
streaming_cleaned_prime_time.head()

,Date,Time,League,Matchup,Services,DayOfWeek
0,3/12/2026,19:00,NBA,"Philadelphia 76ers,Detroit Pistons",[Prime Video],Thursday
1,3/12/2026,19:00,NBA,"Phoenix Suns,Indiana Pacers",[NBA League Pass],Thursday
2,3/12/2026,19:00,NBA,"Washington Wizards,Orlando Magic",[NBA League Pass],Thursday
3,3/12/2026,19:30,NBA,"Brooklyn Nets,Atlanta Hawks",[NBA League Pass],Thursday
4,3/12/2026,19:30,NBA,"Milwaukee Bucks,Miami Heat",[NBA League Pass],Thursday


In [91]:
today = pd.to_datetime("today").date()
streaming_cleaned.to_csv(f"../data/streaming_cleaned_{today}.csv", index=False)